In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # Cambia el 1 por el id de la GPU que quieras usar

In [2]:
import torch
torch.cuda.is_available(), torch.cuda.device_count(), torch.cuda.get_device_name(0)

(True, 1, 'NVIDIA RTX 4500 Ada Generation')

In [3]:
import pandas as pd
from datasets import load_dataset


DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.bfloat16 if DEVICE == 'cuda' else torch.float32
SAMPLE_SIZE = 2500 # Ajustable según capacidad
MAX_LENGTH = 512
SEED = 41
BATCH_SIZE= 4

# Modelos
MODEL_NAMES = {
    'LLaDA': 'GSAI-ML/LLaDA-8B-Base',
    'GPT2': 'gpt2-large',
    'LLaMA': 'NousResearch/Llama-2-7b-hf',
    'BERT': 'bert-base-uncased',
    'RoBERTa': 'roberta-base',
    'GPT3': 'EleutherAI/gpt-neo-2.7B'
}

dataset = load_dataset("yaful/MAGE", split="test")
df_full = dataset.to_pandas()
df_sample = df_full.sample(n=SAMPLE_SIZE, random_state=42)


df_sample['Clase_Real'] = df_sample['label'].apply(lambda x: 'Humano' if x == 1 else 'IA')

print("\nPrimeras filas del DataFrame de muestra:")
print(df_sample[['text', 'Clase_Real', 'label']].head())

balance = df_sample['Clase_Real'].value_counts(normalize=True) * 100

print("\n--- Balance de Clases en la Muestra ---")
print(balance.to_string())

df_sample['Clase_Real_Binaria'] = df_sample['label']


df_sample['text_cleaned'] = df_sample['text'].str.replace('\s+', ' ', regex=True).str.strip()

df_sample = df_sample.dropna(subset=['text_cleaned'])

df_sample.reset_index(drop=True, inplace=True)
df_sample['Texto_ID'] = df_sample.index

texts = df_sample['text_cleaned']
labels = df_sample['Clase_Real_Binaria'].values

# Contenedores de resultados
performance_data = []
all_results = []


Primeras filas del DataFrame de muestra:
                                                    text Clase_Real  label
21764  Never again...never again!!' This place is ter...         IA      0
46722  put the carpet on the floor, they measure it, ...     Humano      1
49245  [substeps] You may do this process before you ...     Humano      1
30867  I believe mandatory minimum laws are unjust, c...     Humano      1
10010  Wales coach Warren Gatland has hailed Shane Wi...         IA      0

--- Balance de Clases en la Muestra ---
Clase_Real
IA        51.24
Humano    48.76


In [4]:
# Función de medición de recursos
def resource_wrapper(fn, *args, device='cuda', verbose=True):
    if device == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    
    process = psutil.Process(os.getpid())
    mem_before = process.memory_info().rss / 1024**3  # en GB

    start_time = time.time()
    try:
        result = fn(*args)
    except Exception as e:
        if verbose:
            print(f"[ERROR] La función {fn.__name__} falló: {e}")
        raise e
    elapsed = time.time() - start_time

    vram_peak = torch.cuda.max_memory_allocated() / 1024**3 if device == 'cuda' else 0.0
    mem_after = process.memory_info().rss / 1024**3
    cpu_mem = mem_after - mem_before

    return result, elapsed, vram_peak, cpu_mem

In [5]:
import os
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import psutil
import pandas as pd
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, AutoModelForMaskedLM

# --- CONFIGURACIÓN ---
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# --- CARGA DE MODELOS ---
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

# LLaDA (Difusión)
tokenizer_llada = AutoTokenizer.from_pretrained(MODEL_NAMES['LLaDA'], trust_remote_code=True)
model_llada = AutoModel.from_pretrained(MODEL_NAMES['LLaDA'], quantization_config=bnb_config, trust_remote_code=True, dtype=DTYPE).eval()
if hasattr(model_llada, "tie_weights"): model_llada.tie_weights()
LLADA_DEVICE = next(model_llada.parameters()).device

# GPT-2
tokenizer_gpt = AutoTokenizer.from_pretrained(MODEL_NAMES['GPT2'])
if tokenizer_gpt.pad_token is None: tokenizer_gpt.pad_token = tokenizer_gpt.eos_token
model_gpt = AutoModelForCausalLM.from_pretrained(MODEL_NAMES['GPT2'], dtype=DTYPE).to(DEVICE).eval()

# LLaMA
tokenizer_llama = AutoTokenizer.from_pretrained(MODEL_NAMES['LLaMA'])
model_llama = AutoModelForCausalLM.from_pretrained(MODEL_NAMES['LLaMA'], quantization_config=bnb_config, dtype=DTYPE).eval()
LLAMA_DEVICE = next(model_llama.parameters()).device

# BERT & RoBERTa (MLM)
tokenizer_bert = AutoTokenizer.from_pretrained(MODEL_NAMES['BERT'])
model_bert = AutoModelForMaskedLM.from_pretrained(MODEL_NAMES['BERT'], torch_dtype=DTYPE).to(DEVICE).eval()

tokenizer_roberta = AutoTokenizer.from_pretrained(MODEL_NAMES['RoBERTa'])
model_roberta = AutoModelForMaskedLM.from_pretrained(MODEL_NAMES['RoBERTa'], torch_dtype=DTYPE).to(DEVICE).eval()

# GPT-3 Proxy
tokenizer_gpt3 = AutoTokenizer.from_pretrained(MODEL_NAMES['GPT3'])
if tokenizer_gpt3.pad_token is None: tokenizer_gpt3.pad_token = tokenizer_gpt3.eos_token
model_gpt3 = AutoModelForCausalLM.from_pretrained(MODEL_NAMES['GPT3'], quantization_config=bnb_config, dtype=DTYPE).eval()
GPT3_DEVICE = next(model_gpt3.parameters()).device

/opt/conda/lib/python3.11/site-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)
2026-01-20 20:17:38.020248: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-20 20:17:38.094008: I tensorflow/core/platform/cpu_feature_guard.cc:210]

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


### SCORES

In [6]:
def batch_llada_scores(texts, model, tokenizer, batch_size, device):
    scores = []
    for i in tqdm(range(0, len(texts), batch_size), desc="LLaDA Scores"):
        batch = texts[i:i+batch_size].tolist()
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH).to(device)
        with torch.no_grad(), torch.amp.autocast("cuda", dtype=DTYPE):
            outputs = model(**inputs)
            logits = outputs.logits[:, :-1, :]
            labels = inputs["input_ids"][:, 1:]
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), labels.reshape(-1), reduction="none").view(labels.shape)
            scores.extend((-loss.mean(dim=1)).cpu().tolist())
    return scores

def batch_autoregressive_scores(texts, model, tokenizer, batch_size, device):
    scores = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Autoregressive Scores"):
        batch = texts[i:i+batch_size].tolist()
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH).to(device)
        with torch.no_grad(), torch.amp.autocast("cuda", dtype=DTYPE):
            outputs = model(**inputs, labels=inputs["input_ids"])
            scores.extend((-outputs.loss.detach().cpu()).repeat(len(batch)).tolist())
    return scores

def batch_mlm_scores(texts, model, tokenizer, batch_size, device):
    scores = []
    for i in tqdm(range(0, len(texts), batch_size), desc="MLM Scores"):
        batch = texts[i:i+batch_size].tolist()
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH).to(device)
        with torch.no_grad(), torch.amp.autocast("cuda", dtype=DTYPE):
            logits = model(**inputs).logits
            log_probs = torch.log_softmax(logits, dim=-1)
            ll = log_probs.gather(2, inputs["input_ids"].unsqueeze(-1)).squeeze(-1)
            ll = (ll * inputs["attention_mask"]).sum(dim=1) / inputs["attention_mask"].sum(dim=1)
            scores.extend(ll.cpu().tolist())
    return scores

In [7]:
# Inicializamos el DataFrame de métricas con la info básica
df_metrics = df_sample[['text_cleaned', 'label']].copy()
df_metrics['Clase_Real'] = df_metrics['label'].apply(lambda x: 'Humano' if x == 1 else 'IA')
df_metrics['Texto_ID'] = df_metrics.index
texts = df_metrics['text_cleaned']

# Diccionario para automatizar ejecución y registro de recursos
scoring_tasks = [
    ("Score_LLaDA", batch_llada_scores, model_llada, tokenizer_llada, LLADA_DEVICE),
    ("Score_GPT", batch_autoregressive_scores, model_gpt, tokenizer_gpt, DEVICE),
    ("Score_LLAMA", batch_autoregressive_scores, model_llama, tokenizer_llama, LLAMA_DEVICE),
    ("Score_GPT3", batch_autoregressive_scores, model_gpt3, tokenizer_gpt3, GPT3_DEVICE),
    ("Score_BERT", batch_mlm_scores, model_bert, tokenizer_bert, DEVICE),
    ("Score_RoBERTa", batch_mlm_scores, model_roberta, tokenizer_roberta, DEVICE),
]

for col_name, func, model, tok, dev in scoring_tasks:
    print(f"\nCalculando {col_name}...")
    result, t, v, ram = resource_wrapper(func, texts, model, tok, BATCH_SIZE, dev)
    df_metrics[col_name] = result
    performance_data.append({
        "Modelo": col_name.split('_')[1],
        "Enfoque": "Sequence Score",
        "Tiempo (s)": t,
        "VRAM Pico (GB)": v,
        "RAM usada (GB)": ram
    })

torch.cuda.empty_cache()
print("\n--- Enfoque 1 completado ---")
df_metrics.head()


Calculando Score_LLaDA...


LLaDA Scores: 100%|██████████| 625/625 [06:38<00:00,  1.57it/s]



Calculando Score_GPT...


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
Autoregressive Scores: 100%|██████████| 625/625 [01:40<00:00,  6.20it/s]



Calculando Score_LLAMA...


...

MLM Scores: 100%|██████████| 625/625 [00:19<00:00, 32.09it/s]



Calculando Score_RoBERTa...


MLM Scores: 100%|██████████| 625/625 [00:20<00:00, 30.91it/s]



--- Enfoque 1 completado ---


,text_cleaned,label,Clase_Real,Texto_ID,Score_LLaDA,Score_GPT,Score_LLAMA,Score_GPT3,Score_BERT,Score_RoBERTa
0,Never again...never again!!' This place is ter...,0,IA,0,-8.285190,-8.636153,-8.940936,-3.870317,-1.068194,-0.114022
1,"put the carpet on the floor, they measure it, ...",1,Humano,1,-3.680638,-8.636153,-8.940936,-3.870317,-2.068663,-0.003787
2,[substeps] You may do this process before you ...,1,Humano,2,-7.870510,-8.636153,-8.940936,-3.870317,-0.886929,-0.234284
3,"I believe mandatory minimum laws are unjust, c...",1,Humano,3,-12.497665,-8.636153,-8.940936,-3.870317,-0.307164,-0.017025
4,Wales coach Warren Gatland has hailed Shane Wi...,0,IA,4,-12.788833,-5.655330,-4.627916,-2.685810,-0.201854,-0.000617


### PAWN 

In [8]:
import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm

def calculate_five_metrics(logits, labels, attention_mask):
    B, T, V = logits.shape
    log_probs = F.log_softmax(logits, dim=-1)
    probs = log_probs.exp()  # Calcular probs UNA VEZ
    
    # 1. Log-prob del token ocurrido
    log_p = log_probs.gather(2, labels.unsqueeze(-1)).squeeze(-1)
    Mlog_prob = log_p
    
    # 2. Entropía
    Mentropy = -(probs * log_probs).sum(dim=-1)
    
    # 3. Máxima log-prob
    Mmax_log_prob, _ = log_probs.max(dim=-1)
    
    # 4. Rank normalizado (CORREGIDO)

    rank = (log_probs > log_p.unsqueeze(-1)).sum(dim=-1).float() + 1.0
    Mrank = rank / V
    
    # 5. Top-p (CORREGIDO)
    prob_occured = probs.gather(2, labels.unsqueeze(-1)).squeeze(-1).unsqueeze(-1)
    Mtop_p = (probs * (probs >= prob_occured)).sum(dim=-1)
    
    metrics = [Mlog_prob, Mentropy, Mmax_log_prob, Mrank, Mtop_p]
    
    seq_len = attention_mask.sum(dim=1).clamp(min=1)
    results = []
    for M in metrics:
        results.append(((M * attention_mask).sum(dim=1) / seq_len).cpu().tolist())
    
    return results

def batch_autoregressive_metrics(texts, model, tokenizer, batch_size, device):
    """
    Calcula las 5 métricas para modelos autoregresivos (LLaDA, GPT, LLaMA, GPT-3)
    """
    all_metrics = [[], [], [], [], []] # 5 listas para las 5 métricas
    
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size].tolist()

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH
        )
        inputs_on_device = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad(), torch.cuda.amp.autocast(dtype=DTYPE):
            outputs = model(**inputs_on_device)
            
            logits = outputs.logits[:, :-1, :]        # (B, T-1, V)
            labels = inputs_on_device["input_ids"][:, 1:] # (B, T-1)

            attention_mask = inputs_on_device["attention_mask"][:, 1:] # (B, T-1)
            
            metrics = calculate_five_metrics(logits, labels, attention_mask)
            
            for j in range(5):
                all_metrics[j].extend(metrics[j])
                
    return all_metrics
def calculate_five_metrics_mlm(logits, labels, attention_mask):
    """
    Calcula las 5 métricas PAWN para modelos MLM.
    Para LLaDA en modo MLM: evalúa cada posición de la secuencia.
    """
    B, T, V = logits.shape
    
    log_probs = F.log_softmax(logits, dim=-1)  # (B, T, V)
    probs = torch.exp(log_probs)
    
    # 1. Log-prob del token real
    log_prob_occured = log_probs.gather(
        2, labels.unsqueeze(-1)
    ).squeeze(-1)  # (B, T)
    
    Mlog_prob = log_prob_occured
    
    # 2. Entropía
    entropy = -(probs * log_probs).sum(dim=-1)  # (B, T)
    Mentropy = entropy
    
    # 3. Max log-prob
    Mmax_log_prob, _ = log_probs.max(dim=-1)  # (B, T)
    
    # 4. Rank (CORREGIDO - posición ordinal normalizada)
    log_prob_occured_val = log_prob_occured.unsqueeze(-1)  # (B, T, 1)
    rank = (log_probs > log_prob_occured_val).sum(dim=-1).float() + 1.0
    Mrank = rank / V  # Normalizar
    
    # 5. Top-p
    prob_occured = probs.gather(2, labels.unsqueeze(-1)).squeeze(-1).unsqueeze(-1)
    Mtop_p = (probs * (probs >= prob_occured)).sum(dim=-1)
    
    masked_metrics = [Mlog_prob, Mentropy, Mmax_log_prob, Mrank, Mtop_p]
    results = []
    sequence_lengths = attention_mask.sum(dim=1).float().clamp(min=1)  # (B,)
    
    for M in masked_metrics:
        M_masked = M * attention_mask
        M_sum_per_seq = M_masked.sum(dim=1)
        M_avg_per_seq = M_sum_per_seq / sequence_lengths
        results.append(M_avg_per_seq.cpu().tolist())
    
    return results


def batch_mlm_metrics(texts, model, tokenizer, batch_size, device):
    """
    Calcula las 5 métricas PAWN para LLaDA en modo MLM.
    LLaDA puede funcionar como MLM bidireccional.
    """
    all_metrics = [[], [], [], [], []]
    vocab_size = model.config.vocab_size
    
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size].tolist()
        
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH
        ).to(device)
        
        input_ids = inputs["input_ids"]
        attention_mask = inputs["attention_mask"]
        
        # CRÍTICO: Clamp de seguridad
        input_ids = input_ids.clamp(0, vocab_size - 1)
        
        with torch.no_grad(), torch.amp.autocast("cuda", dtype=DTYPE):
            # LLaDA puede retornar logits para todas las posiciones
            outputs = model(**inputs)
            
            # Si LLaDA retorna logits de forma (B, T, V), úsalos directamente
            logits = outputs.logits  # (B, T, V)
            
            # Calcular las 5 métricas
            metrics = calculate_five_metrics_mlm(logits, input_ids, attention_mask)
            
            for j in range(5):
                all_metrics[j].extend(metrics[j])
    
    return all_metrics

def calculate_five_metrics_diffusion(logits, labels, mask_positions):
    """
    Calcula las 5 métricas PAWN solo en las posiciones ENMASCARADAS.
    Basado en la lógica de reconstrucción de Language Diffusion.
    """
    B, T, V = logits.shape
    
    # Trabajamos con log_softmax para estabilidad numérica
    log_probs = F.log_softmax(logits, dim=-1)
    probs = torch.exp(log_probs)
    
    # 1. Log-probabilidad del token real (¿Qué tan bien reconstruye el modelo el texto original?)
    log_p = log_probs.gather(2, labels.unsqueeze(-1)).squeeze(-1)
    
    # 2. Entropía (Incertidumbre del modelo en las zonas enmascaradas)
    entropy = -(probs * log_probs).sum(dim=-1)
    
    # 3. Max log-prob (Confianza máxima en la predicción)
    max_log_p, _ = log_probs.max(dim=-1)
    
    # 4. Rank normalizado (Posición ordinal del token real)
    # Calculamos cuántos tokens tienen mayor probabilidad que el token real
    rank = (log_probs > log_p.unsqueeze(-1)).sum(dim=-1).float() + 1.0
    rank_norm = rank / V
    
    # 5. Top-p (Masa de probabilidad acumulada necesaria para llegar al token real)
    # Indica qué tan "esperada" era la palabra en ese contexto
    prob_occured = probs.gather(2, labels.unsqueeze(-1)).squeeze(-1).unsqueeze(-1)
    top_p = (probs * (probs >= prob_occured)).sum(dim=-1)
    
    mask_float = mask_positions.float()
    num_masked = mask_float.sum(dim=1).clamp(min=1)
    
    results = []
    # Iteramos sobre las 5 métricas calculadas
    for M in [log_p, entropy, max_log_p, rank_norm, top_p]:
        # Filtramos para obtener el promedio SOLO de los tokens que fueron enmascarados
        # Esto es lo que mide la capacidad de "denoising" o reconstrucción.
        avg_val = (M * mask_float).sum(dim=1) / num_masked
        results.append(avg_val.cpu().tolist())
        
    return results

def batch_diffusion_metrics(texts, model, tokenizer, batch_size, device, mask_ratio=0.35, num_samples=10):
    """
    Implementa el proceso de scoring por difusión para LLaDA.
    Aumentamos el mask_ratio a 0.20 para forzar al modelo a usar más contexto.
    Reducimos num_samples a 3 para balancear velocidad y estabilidad.
    """
    all_metrics = [[] for _ in range(5)]
    
    # Identificar token de máscara correcto
    if hasattr(tokenizer, 'mask_token_id') and tokenizer.mask_token_id is not None:
        mask_id = tokenizer.mask_token_id
    else:
        # Fallback para modelos que no tienen [MASK] definido explícitamente
        mask_id = tokenizer.vocab_size - 1 

    model.eval()
    
    # Determinamos el tipo de dato para autocast (bfloat16 para GPUs modernas como RTX 4500 Ada)
    dtype = torch.bfloat16 if device == 'cuda' else torch.float32

    for i in tqdm(range(0, len(texts), batch_size), desc="LLaDA Diffusion Scoring"):
        batch_texts = texts[i:i+batch_size].tolist()
        # Acumulador para las muestras estocásticas de cada batch
        batch_accum = [[] for _ in range(5)]
        
        # Realizamos varias pasadas con diferentes máscaras para obtener un promedio robusto (Monte Carlo)
        for _ in range(num_samples):
            inputs = tokenizer(
                batch_texts, 
                return_tensors="pt", 
                padding=True, 
                truncation=True, 
                max_length=512
            ).to(device)
            
            input_ids = inputs["input_ids"]
            att_mask = inputs["attention_mask"]
            B, T = input_ids.shape
            
            # Generamos máscara aleatoria excluyendo el padding
            mask_probs = torch.full((B, T), mask_ratio, device=device) * att_mask.float()
            
            # Opcional: Evitar enmascarar tokens especiales (si el tokenizer los tiene definidos)
            if hasattr(tokenizer, 'all_special_ids'):
                for special_id in tokenizer.all_special_ids:
                    mask_probs[input_ids == special_id] = 0.0

            mask_pos = torch.bernoulli(mask_probs).bool()
            
            # Garantizar que al menos un token esté enmascarado por secuencia
            for j in range(B):
                if not mask_pos[j].any():
                    valid_indices = att_mask[j].nonzero(as_tuple=True)[0]
                    if len(valid_indices) > 0:
                        random_idx = valid_indices[torch.randint(0, len(valid_indices), (1,))]
                        mask_pos[j, random_idx] = True

            # Crear la versión "corrupta" del texto
            corrupted_ids = input_ids.clone()
            corrupted_ids[mask_pos] = mask_id
            
            with torch.no_grad(), torch.amp.autocast("cuda", dtype=dtype):
                # Predicción bidireccional: el modelo intenta adivinar los tokens en las máscaras
                outputs = model(input_ids=corrupted_ids, attention_mask=att_mask)
                
                # Extraer métricas solo de las posiciones que el modelo tuvo que reconstruir
                sample_m = calculate_five_metrics_diffusion(outputs.logits, input_ids, mask_pos)
                
                for m_idx in range(5):
                    batch_accum[m_idx].append(sample_m[m_idx])
        
        # Promediar las muestras para reducir el ruido de la selección aleatoria de máscaras
        for m_idx in range(5):
            avg_res = np.array(batch_accum[m_idx]).mean(axis=0)
            all_metrics[m_idx].extend(avg_res.tolist())
            
    return all_metrics

In [9]:
metrics_names = ['Mlog_prob', 'Mentropy', 'Mmax_log_prob', 'Mrank', 'Mtop_p']

# Tareas de extracción PAWN
pawn_tasks = [
    ("LLaDA", batch_diffusion_metrics, model_llada, tokenizer_llada, LLADA_DEVICE), # Usa tu función diffusion
    ("GPT", batch_autoregressive_metrics, model_gpt, tokenizer_gpt, DEVICE),
    ("LLaMA", batch_autoregressive_metrics, model_llama, tokenizer_llama, LLAMA_DEVICE),
    ("GPT3", batch_autoregressive_metrics, model_gpt3, tokenizer_gpt3, GPT3_DEVICE),
    ("BERT", batch_mlm_metrics, model_bert, tokenizer_bert, DEVICE),
    ("RoBERTa", batch_mlm_metrics, model_roberta, tokenizer_roberta, DEVICE),
]

for mod_name, func, model, tok, dev in pawn_tasks:
    print(f"\nCalculando métricas PAWN para {mod_name}...")
    # Registramos recursos
    pawn_results, t, v, ram = resource_wrapper(func, texts, model, tok, BATCH_SIZE, dev)
    
    # Guardamos las 5 métricas en el DataFrame
    for idx, m_name in enumerate(metrics_names):
        df_metrics[f'{m_name}_{mod_name}'] = pawn_results[idx]
        
    # Añadimos a la tabla de rendimiento
    performance_data.append({
        "Modelo": mod_name,
        "Enfoque": "PAWN Extraction",
        "Tiempo (s)": t,
        "VRAM Pico (GB)": v,
        "RAM usada (GB)": ram
    })

torch.cuda.empty_cache()
print("\n--- Enfoque 2: PAWN Metrics completado ---")
df_metrics.filter(like='Mlog_prob').head()


Calculando métricas PAWN para LLaDA...


LLaDA Diffusion Scoring: 100%|██████████| 625/625 [1:35:55<00:00,  9.21s/it]



Calculando métricas PAWN para GPT...


/tmp/ipykernel_14926/712804160.py:57: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(dtype=DTYPE):
100%|██████████| 625/625 [01:50<00:00,  5.68it/s]



Calculando métricas PAWN para LLaMA...


...

100%|██████████| 625/625 [00:24<00:00, 25.02it/s]



Calculando métricas PAWN para RoBERTa...


100%|██████████| 625/625 [00:29<00:00, 21.15it/s]



--- Enfoque 2: PAWN Metrics completado ---


,Mlog_prob_LLaDA,Mlog_prob_GPT,Mlog_prob_LLaMA,Mlog_prob_GPT3,Mlog_prob_BERT,Mlog_prob_RoBERTa
0,-7.858474,-3.580119,-3.659339,-4.055174,-1.068194,-0.114022
1,-6.560389,-3.744196,-3.722310,-3.708690,-2.068663,-0.003787
2,-7.536360,-3.750089,-3.931632,-3.865711,-0.886929,-0.234284
3,-7.644753,-2.831601,-2.139618,-2.876837,-0.307164,-0.017025
4,-7.321009,-1.784918,-1.318048,-1.838927,-0.201854,-0.000617


### EMBEDINGS

In [10]:
def extract_cls_embeddings_autoregressive(texts, model, tokenizer, device, batch_size=8):
    """
    Extrae embeddings de la ÚLTIMA posición (equivalente a CLS en modelos autoregresivos).
    Estos modelos generan representaciones causales, por lo que el último token
    tiene información de toda la secuencia.
    """
    all_embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Extrayendo embeddings autoregresivos"):
        batch = texts[i:i+batch_size].tolist()
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad(), torch.amp.autocast('cuda', dtype=DTYPE):
            outputs = model(**inputs, output_hidden_states=True)
            hidden_states = outputs.hidden_states[-1]  # última capa
            
            # Extraer embedding del último token no-padding de cada secuencia
            attention_mask = inputs['attention_mask']
            seq_lengths = attention_mask.sum(dim=1) - 1  # índice del último token
            
            batch_embeddings = []
            for j, seq_len in enumerate(seq_lengths):
                # Último token con información de toda la secuencia
                embedding = hidden_states[j, seq_len, :].float().cpu().numpy()
                batch_embeddings.append(embedding)
            
            all_embeddings.extend(batch_embeddings)
    
    return np.array(all_embeddings)


def extract_cls_embeddings_mlm(texts, model, tokenizer, device, batch_size=8):
    """
    Extrae embeddings del token [CLS] o equivalente para modelos MLM.
    BERT y RoBERTa tienen un token especial al inicio que agrega contexto bidireccional.
    """
    all_embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Extrayendo embeddings MLM"):
        batch = texts[i:i+batch_size].tolist()
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad(), torch.amp.autocast('cuda', dtype=DTYPE):
            outputs = model(**inputs, output_hidden_states=True)
            hidden_states = outputs.hidden_states[-1]  # última capa
            
            # Token [CLS] está en la posición 0
            cls_embeddings = hidden_states[:, 0, :].float().cpu().numpy()
            all_embeddings.extend(cls_embeddings)
    
    return np.array(all_embeddings)


def extract_cls_embeddings_diffusion(texts, model, tokenizer, device, batch_size=8):
    """
    Extrae embeddings de LLaDA (modelo de difusión).
    LLaDA procesa bidireccionalmente, similar a BERT, por lo que usamos
    el promedio de todos los tokens como representación global.
    
    Alternativa: pooling del primer token o mean pooling de toda la secuencia.
    """
    all_embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Extrayendo embeddings difusión"):
        batch = texts[i:i+batch_size].tolist()
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad(), torch.amp.autocast('cuda', dtype=DTYPE):
            outputs = model(**inputs, output_hidden_states=True)
            hidden_states = outputs.hidden_states[-1]  # última capa
            
            # Mean pooling sobre tokens válidos (excluyendo padding)
            attention_mask = inputs['attention_mask'].unsqueeze(-1)
            masked_hidden = hidden_states * attention_mask
            sum_hidden = masked_hidden.sum(dim=1)
            count = attention_mask.sum(dim=1).clamp(min=1)
            mean_embedding = (sum_hidden / count).float().cpu().numpy()
            
            all_embeddings.extend(mean_embedding)
    
    return np.array(all_embeddings)


In [11]:
embedding_tasks = [
    ("LLaDA", extract_cls_embeddings_diffusion, model_llada, tokenizer_llada, LLADA_DEVICE),
    ("GPT2", extract_cls_embeddings_autoregressive, model_gpt, tokenizer_gpt, DEVICE),
    ("LLAMA", extract_cls_embeddings_autoregressive, model_llama, tokenizer_llama, LLAMA_DEVICE),
    ("BERT", extract_cls_embeddings_mlm, model_bert, tokenizer_bert, DEVICE),
    ("RoBERTa", extract_cls_embeddings_mlm, model_roberta, tokenizer_roberta, DEVICE)
]

for mod_name, func, model, tok, dev in embedding_tasks:
    print(f"\n[INFO] Extrayendo embeddings para {mod_name}...")
    
    # 1. Ejecutamos la extracción y medimos recursos (Tiempo, VRAM, RAM)
    # Usamos BATCH_SIZE=4 como en tus otros scripts para evitar errores de memoria
    embs_matrix, t, v, ram = resource_wrapper(func, texts, model, tok, dev, BATCH_SIZE)
    
    # 2. Guardamos los embeddings en el DataFrame
    # Convertimos la matriz numpy a una lista para que cada celda sea un vector individual
    df_metrics[f'Embedding_{mod_name}'] = list(embs_matrix)
    
    # 3. Registramos el rendimiento en la tabla global de recursos
    performance_data.append({
        "Modelo": mod_name,
        "Enfoque": "CLS Embedding Extraction",
        "Tiempo (s)": t,
        "VRAM Pico (GB)": v,
        "RAM usada (GB)": ram
    })

# Limpieza de caché de GPU
torch.cuda.empty_cache()

print("\n" + "="*50)
print("ESTADO DEL DATAFRAME INTEGRADO")
print("="*50)
# Mostramos las nuevas columnas creadas junto a las anteriores
cols_interes = ['Texto_ID'] + [c for c in df_metrics.columns if 'Embedding' in c or 'Score' in c or 'Mlog_prob' in c]
print(df_metrics[cols_interes].head())

# Guardado intermedio del progreso
df_metrics.to_csv('df_all_features_combined.csv', index=False)
print("\n✓ Todas las métricas (Scores, PAWN y Embeddings) guardadas en 'df_all_features_combined.csv'")


[INFO] Extrayendo embeddings para LLaDA...


Extrayendo embeddings difusión: 100%|██████████| 625/625 [06:32<00:00,  1.59it/s]



[INFO] Extrayendo embeddings para GPT2...


Extrayendo embeddings autoregresivos: 100%|██████████| 625/625 [01:39<00:00,  6.31it/s]



[INFO] Extrayendo embeddings para LLAMA...


...


[INFO] Extrayendo embeddings para BERT...


Extrayendo embeddings MLM: 100%|██████████| 625/625 [00:18<00:00, 33.37it/s]



[INFO] Extrayendo embeddings para RoBERTa...


Extrayendo embeddings MLM: 100%|██████████| 625/625 [00:18<00:00, 33.78it/s]



ESTADO DEL DATAFRAME INTEGRADO
   Texto_ID  Score_LLaDA  Score_GPT  Score_LLAMA  Score_GPT3  Score_BERT  \
0         0    -8.285190  -8.636153    -8.940936   -3.870317   -1.068194   
1         1    -3.680638  -8.636153    -8.940936   -3.870317   -2.068663   
2         2    -7.870510  -8.636153    -8.940936   -3.870317   -0.886929   
3         3   -12.497665  -8.636153    -8.940936   -3.870317   -0.307164   
4         4   -12.788833  -5.655330    -4.627916   -2.685810   -0.201854   

   Score_RoBERTa  Mlog_prob_LLaDA  Mlog_prob_GPT  Mlog_prob_LLaMA  \
0      -0.114022        -7.858474      -3.580119        -3.659339   
1      -0.003787        -6.560389      -3.744196        -3.722310   
2      -0.234284        -7.536360      -3.750089        -3.931632   
3      -0.017025        -7.644753      -2.831601        -2.139618   
4      -0.000617        -7.321009      -1.784918        -1.318048   

   Mlog_prob_GPT3  Mlog_prob_BERT  Mlog_prob_RoBERTa  \
0       -4.055174       -1.068194       

In [12]:
df = pd.DataFrame(performance_data)

df.to_csv('df_performance_models_proachs.csv', index=False)
df.head()

,Modelo,Enfoque,Tiempo (s),VRAM Pico (GB),RAM usada (GB)
0,LLaDA,Sequence Score,398.081016,15.534767,0.058983
1,GPT,Sequence Score,100.754739,14.994935,0.010731
2,LLAMA,Sequence Score,359.512608,15.877870,0.003891
3,GPT3,Sequence Score,210.824698,16.800073,0.007000
4,BERT,Sequence Score,19.478747,13.941291,0.006760


### CLASIFICACIÓN

In [13]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, recall_score, precision_score
from sklearn.calibration import CalibratedClassifierCV
import xgboost as xgb

# =============================================================================
# 0. FUNCIÓN DE EVALUACIÓN (idéntica a Embedings.ipynb)
# =============================================================================
def evaluate_classifier(y_true, y_pred_probs):
    y_pred = (y_pred_probs >= 0.5).astype(int)
    metrics = {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0),
    }
    try:
        metrics['ROC-AUC'] = roc_auc_score(y_true, y_pred_probs)
    except:
        metrics['ROC-AUC'] = np.nan
    return metrics

import ast

def parse_embedding_safe(x):
    """
    Convierte un string tipo '[0.1 0.2 ...]' o '[0.1, 0.2, ...]'
    en np.ndarray float32 de forma segura.
    """
    if isinstance(x, np.ndarray):
        return x
    if isinstance(x, str):
        # Normalizamos espacios -> comas
        x = x.replace('\n', ' ').replace('  ', ' ')
        if ',' not in x:
            x = x.replace(' ', ', ')
        return np.array(ast.literal_eval(x), dtype=np.float32)
    raise ValueError(f"Tipo inesperado en embedding: {type(x)}")

# =============================================================================
# 1. ARQUITECTURA Y MOTOR MLP (PAWN)
# =============================================================================
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class DeepMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 1024), nn.BatchNorm1d(1024), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 128), nn.ReLU(), nn.Linear(128, 2)
        )
    def forward(self, x): 
        return self.net(x)

def train_eval_mlp_full(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=SEED
    )
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    model = DeepMLP(X.shape[1]).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-5)
    criterion = nn.CrossEntropyLoss()

    loader = DataLoader(
        TensorDataset(torch.from_numpy(X_train).float(), torch.from_numpy(y_train).long()),
        batch_size=32, shuffle=True
    )

    for _ in range(100):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            criterion(model(xb), yb).backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        logits = model(torch.from_numpy(X_test).float().to(DEVICE))
        probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()

    return evaluate_classifier(y_test, probs)

# =============================================================================
# 2. CLASIFICACIÓN MACRO (SCORE / PAWN / EMBEDDINGS)
# =============================================================================
df = pd.read_csv("df_all_features_combined.csv")
y_all = df["label"].values
results_master = []

models_list = ['LLaDA', 'GPT2', 'LLAMA', 'GPT3', 'BERT', 'RoBERTa']
metrics_pawn_names = ['Mlog_prob', 'Mentropy', 'Mmax_log_prob', 'Mrank', 'Mtop_p']

for m in models_list:
    print(f"\nEvaluando: {m}")

    # -------------------------------------------------------------------------
    # A) SCORE
    # -------------------------------------------------------------------------
    s_col = f"Score_{m}"
    if s_col in df.columns:
        X_s = df[[s_col]].values
        clfs = {
            'LogReg': LogisticRegression(max_iter=2000),
            'RandomForest': RandomForestClassifier(n_estimators=300, random_state=SEED),
            'XGBoost': xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, eval_metric="logloss", random_state=SEED),
            'XGB_Calib': CalibratedClassifierCV(
                xgb.XGBClassifier(eval_metric="logloss"), method="isotonic", cv=3
            )
        }
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
        for name, clf in clfs.items():
            cv_res = cross_validate(
                clf, X_s, y_all, cv=cv,
                scoring=['roc_auc', 'accuracy', 'f1', 'recall', 'precision']
            )
            results_master.append({
                'Modelo': m, 'Aprox': 'Score', 'Clf': name,
                'ROC-AUC': np.mean(cv_res['test_roc_auc']),
                'Accuracy': np.mean(cv_res['test_accuracy']),
                'F1': np.mean(cv_res['test_f1']),
                'Recall': np.mean(cv_res['test_recall']),
                'Precision': np.mean(cv_res['test_precision'])
            })

    # -------------------------------------------------------------------------
    # B) PAWN
    # -------------------------------------------------------------------------
    p_cols = [f"{met}_{m}" for met in metrics_pawn_names]
    if all(c in df.columns for c in p_cols):
        X_p = df[p_cols].values

        # PAWN + LogReg
        X_tr, X_te, y_tr, y_te = train_test_split(
            X_p, y_all, test_size=0.2, stratify=y_all, random_state=SEED
        )
        sc = StandardScaler()
        clf_lr = LogisticRegression(
            max_iter=1000, class_weight='balanced', random_state=SEED
        ).fit(sc.fit_transform(X_tr), y_tr)
        p_lr = clf_lr.predict_proba(sc.transform(X_te))[:, 1]

        results_master.append({
            'Modelo': m, 'Aprox': 'PAWN', 'Clf': 'LogReg',
            **evaluate_classifier(y_te, p_lr)
        })

        # PAWN + DeepMLP
        results_master.append({
            'Modelo': m, 'Aprox': 'PAWN', 'Clf': 'DeepMLP',
            **train_eval_mlp_full(X_p, y_all)
        })

# -------------------------------------------------------------------------
# C) EMBEDDINGS (extracción + clasificación DIRECTA, sin guardar)
# -------------------------------------------------------------------------
print("\n" + "="*80)
print("EXTRACCIÓN DE EMBEDDINGS Y CLASIFICACIÓN (DIRECTO)")
print("="*80)

# Diccionario que mapea modelo → función de extracción + objetos
embedding_configs = {
    'LLaDA': {
        'extractor': lambda: extract_cls_embeddings_diffusion(
            texts, model_llada, tokenizer_llada, LLADA_DEVICE
        )
    },
    'GPT2': {
        'extractor': lambda: extract_cls_embeddings_autoregressive(
            texts, model_gpt, tokenizer_gpt, DEVICE
        )
    },
    'LLAMA': {
        'extractor': lambda: extract_cls_embeddings_autoregressive(
            texts, model_llama, tokenizer_llama, LLAMA_DEVICE
        )
    },
    'BERT': {
        'extractor': lambda: extract_cls_embeddings_mlm(
            texts, model_bert, tokenizer_bert, DEVICE
        )
    },
    'RoBERTa': {
        'extractor': lambda: extract_cls_embeddings_mlm(
            texts, model_roberta, tokenizer_roberta, DEVICE
        )
    }
}

for m, cfg in embedding_configs.items():
    print(f"\n[EMB] {m} - Extrayendo embeddings...")
    
    # 1. Extracción REAL del embedding (np.ndarray)
    X_e = cfg['extractor']()
    print(f"   Shape: {X_e.shape}")

    # 2. Split (idéntico a Embedings.ipynb)
    X_train, X_test, y_train, y_test = train_test_split(
        X_e, y_all, test_size=0.2, random_state=SEED, stratify=y_all
    )

    # 3. Escalado
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # 4. Clasificador lineal
    clf = LogisticRegression(
        max_iter=1000,
        random_state=SEED,
        class_weight='balanced'
    )
    clf.fit(X_train_scaled, y_train)

    # 5. Predicción
    y_pred_probs = clf.predict_proba(X_test_scaled)[:, 1]

    # 6. Métricas
    metrics = evaluate_classifier(y_test, y_pred_probs)

    print(f"   Resultados: {metrics}")

    # 7. Inserción DIRECTA en la macro-tabla
    results_master.append({
        'Modelo': m,
        'Aprox': 'Embedding',
        'Clf': 'LogReg',
        'ROC-AUC': metrics['ROC-AUC'],
        'Accuracy': metrics['Accuracy'],
        'F1': metrics['F1'],
        'Recall': metrics['Recall'],
        'Precision': metrics['Precision']
    })

# =============================================================================
# 3. PRESENTACIÓN DE RESULTADOS
# =============================================================================
df_final = pd.DataFrame(results_master)
print("\n" + "="*120)
print("MACRO TABLA DE RESULTADOS (COMPARATIVA FINAL)")
print("="*120)
display(df_final.set_index(['Modelo', 'Aprox', 'Clf']).round(4))



Evaluando: LLaDA

Evaluando: GPT2

Evaluando: LLAMA


/home/jhuertas/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/jhuertas/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/jhuertas/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



Evaluando: GPT3


/home/jhuertas/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/jhuertas/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/jhuertas/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/jhuertas/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1471: Un


Evaluando: BERT

Evaluando: RoBERTa

EXTRACCIÓN DE EMBEDDINGS Y CLASIFICACIÓN (DIRECTO)

[EMB] LLaDA - Extrayendo embeddings...


...

Extrayendo embeddings MLM: 100%|██████████| 313/313 [00:20<00:00, 15.38it/s]


   Shape: (2500, 768)
   Resultados: {'Accuracy': 0.652, 'Precision': 0.6434426229508197, 'Recall': 0.6434426229508197, 'F1': 0.6434426229508197, 'ROC-AUC': 0.7082959784836066}

[EMB] RoBERTa - Extrayendo embeddings...


Extrayendo embeddings MLM: 100%|██████████| 313/313 [00:20<00:00, 15.26it/s]


   Shape: (2500, 768)
   Resultados: {'Accuracy': 0.728, 'Precision': 0.7093023255813954, 'Recall': 0.75, 'F1': 0.7290836653386455, 'ROC-AUC': 0.7833792264344261}

MACRO TABLA DE RESULTADOS (COMPARATIVA FINAL)


ROC-AUC  Accuracy      F1  Recall  Precision
Modelo  Aprox     Clf                                                       
LLaDA   Score     LogReg         0.5304    0.5168  0.4241  0.3650     0.5071
                  RandomForest   0.5211    0.5060  0.4881  0.4840     0.4931
                  XGBoost        0.5545    0.5388  0.5381  0.5513     0.5260
                  XGB_Calib      0.5353    0.5132  0.5549  0.6316     0.4999
        PAWN      LogReg         0.6031    0.5560  0.5316  0.5164     0.5478
                  DeepMLP        0.7038    0.6300  0.6380  0.6680     0.6105
LLAMA   Score     LogReg         0.5070    0.5104  0.0423  0.0246     0.1947
                  RandomForest   0.4916    0.4920  0.4726  0.4676     0.4787
                  XGBoost        0.5042    0.5060  0.4730  0.4553     0.4931
                  XGB_Calib      0.5027    0.5064  0.3555  0.2822     0.4953
GPT3    Score     LogReg         0.4825    0.5120  0.0032  0.0016     0.0800
                  RandomForest   0.4896    0.4816  0.4581  0.4504     0.4680
                  XGBoost        0.4948    0.5036  0.4665  0.4455     0.4908
                  XGB_Calib      0.4793    0.5092  0.1307  0.0828     0.3817
        PAWN      LogReg         0.6207    0.5580  0.5462  0.5451     0.5473
                  DeepMLP        0.7798    0.7060  0.7470  0.8893     0.6439
BERT    Score     LogReg         0.5263    0.5140  0.3414  0.2592     0.5034
                  RandomForest   0.4815    0.4944  0.4805  0.4808     0.4812
                  XGBoost        0.5036    0.5016  0.4673  0.4496     0.4881
                  XGB_Calib      0.4970    0.4984  0.2209  0.1551     0.4705
        PAWN      LogReg         0.5694    0.5440  0.4673  0.4098     0.5435
                  DeepMLP        0.6138    0.5900  0.6279  0.7090     0.5635
RoBERTa Score     LogReg         0.5897    0.5560  0.3911  0.2929     0.5910
                  RandomForest   0.5163    0.5120  0.4961  0.4938     0.4991
                  XGBoost        0.5512    0.5388  0.5110  0.4938     0.5296
                  XGB_Calib      0.5460    0.5296  0.4181  0.3536     0.5492
        PAWN      LogReg         0.6198    0.6120  0.5571  0.5000     0.6289
                  DeepMLP        0.6510    0.6180  0.5996  0.5861     0.6137
LLaDA   Embedding LogReg         0.9156    0.8560  0.8548  0.8689     0.8413
GPT2    Embedding LogReg         0.7478    0.6760  0.6721  0.6803     0.6640
LLAMA   Embedding LogReg         0.6832    0.6060  0.6917  0.9057     0.5595
BERT    Embedding LogReg         0.7083    0.6520  0.6434  0.6434     0.6434
RoBERTa Embedding LogReg         0.7834    0.7280  0.7291  0.7500     0.7093

In [15]:
df_final.to_csv('df_results_models_proachs.csv', index=False)

,Modelo,Aprox,Clf,ROC-AUC,Accuracy,F1,Recall,Precision
0,LLaDA,Score,LogReg,0.530436,0.5168,0.424130,0.365034,0.507125
1,LLaDA,Score,RandomForest,0.521060,0.5060,0.488078,0.484042,0.493112
2,LLaDA,Score,XGBoost,0.554464,0.5388,0.538132,0.551265,0.526048
3,LLaDA,Score,XGB_Calib,0.535307,0.5132,0.554944,0.631623,0.499931
4,LLaDA,PAWN,LogReg,0.603099,0.5560,0.531646,0.516393,0.547826
5,LLaDA,PAWN,DeepMLP,0.703829,0.6300,0.637965,0.668033,0.610487
6,LLAMA,Score,LogReg,0.506975,0.5104,0.042287,0.024590,0.194667
7,LLAMA,Score,RandomForest,0.491581,0.4920,0.472618,0.467594,0.478707
8,LLAMA,Score,XGBoost,0.504219,0.5060,0.473047,0.455292,0.493085
9,LLAMA,Score,XGB_Calib,0.502675,0.5064,0.355528,0.282220,0.495346


In [17]:
df

,text_cleaned,label,Clase_Real,Texto_ID,Score_LLaDA,Score_GPT,Score_LLAMA,Score_GPT3,Score_BERT,Score_RoBERTa,...,Mlog_prob_RoBERTa,Mentropy_RoBERTa,Mmax_log_prob_RoBERTa,Mrank_RoBERTa,Mtop_p_RoBERTa,Embedding_LLaDA,Embedding_GPT2,Embedding_LLAMA,Embedding_BERT,Embedding_RoBERTa
0,Never again...never again!!' This place is ter...,0,IA,0,-8.285190,-8.636153,-8.940936,-3.870317,-1.068194,-0.114022,...,-0.114022,0.086216,-0.020272,0.000021,0.984737,[-0.38913754 -0.4282905 -0.5653935 ... -1.18...,[-0.5711137 -0.23565297 -1.4957054 ... 0.95...,[ 0.26367188 -0.03466797 -0.03088379 ... -0.12...,[ 1.07997335e-01 8.07870701e-02 2.26925626e-...,[-4.15497683e-02 3.41629088e-02 -2.81518325e-...
1,"put the carpet on the floor, they measure it, ...",1,Humano,1,-3.680638,-8.636153,-8.940936,-3.870317,-2.068663,-0.003787,...,-0.003787,0.026394,-0.003787,0.000020,0.996308,[-0.7744284 -0.030733 -1.2288603 ... -1.37603...,[ 0.12539466 -0.4477669 -0.30969116 ... 0.08...,[ 0.26367188 -0.03466797 -0.03088379 ... -0.12...,[-1.15959227e-01 5.63281238e-01 3.85829955e-...,[-9.24953222e-02 8.36356580e-02 -3.10991611e-...
2,[substeps] You may do this process before you ...,1,Humano,2,-7.870510,-8.636153,-8.940936,-3.870317,-0.886929,-0.234284,...,-0.234284,0.052307,-0.014869,0.000128,0.998585,[-1.1252631 -1.0051453 -0.3350762 ... -1.05646...,[-0.71747065 -0.07337198 -0.6382847 ... 0.13...,[ 0.26367188 -0.03466797 -0.03088379 ... -0.12...,[-4.17813987e-01 -3.75171691e-01 -2.16270741e-...,[-7.85473213e-02 1.22823365e-01 -6.08439744e-...
3,"I believe mandatory minimum laws are unjust, c...",1,Humano,3,-12.497665,-8.636153,-8.940936,-3.870317,-0.307164,-0.017025,...,-0.017025,0.054208,-0.016671,0.000020,0.991334,[ 0.2978003 0.0839352 -0.8172775 ... -1.03977...,[-0.564155 0.27531508 -0.36945328 ... 0.63...,[-1.25 1.609375 1.7890625 ... -0.07...,[-7.06142545e-01 -4.92070735e-01 -1.03970778e+...,[-7.21956789e-02 1.32712811e-01 -3.97389149e-...
4,Wales coach Warren Gatland has hailed Shane Wi...,0,IA,4,-12.788833,-5.655330,-4.627916,-2.685810,-0.201854,-0.000617,...,-0.000617,0.004569,-0.000617,0.000020,0.999389,[-0.04816202 -0.15971391 0.29767957 ... -0.33...,[-0.0393448 -0.08812084 1.0711623 ... -0.04...,[ 1.78125 0.40234375 -0.66015625 ... 0.20...,[-4.65582937e-01 -1.77734464e-01 -4.08333153e-...,[-5.79304919e-02 9.07322541e-02 1.75109804e-...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2495,My gaze slowly slides to the bottom corner of ...,1,Humano,2495,-12.038179,-7.443975,-7.316145,-2.499406,-0.223635,-0.155842,...,-0.155842,0.099855,-0.036130,0.001186,0.992704,[-0.45695877 -1.4182454 -0.9570299 ... -0.67...,[-0.50546384 -0.38944086 -1.1876302 ... 0.13...,[-0.30273438 -0.4921875 3.25 ... -0.87...,[-2.16688171e-01 -2.46478077e-02 2.62543857e-...,[-7.44552836e-02 1.08536273e-01 -3.90146160e-...
2496,Just wanted to highly recommend John at this l...,0,IA,2496,-13.347786,-2.644980,-2.242123,-2.618173,-0.103470,-0.000405,...,-0.000405,0.003335,-0.000405,0.000020,0.999598,[-1.7630024 -1.3053155 0.24823561 ... -2.27...,[ 0.2861854 -0.3256478 0.40068305 ... 0.97...,[ 1.0390625 1.8125 -0.921875 ... -1.28...,[-2.01839864e-01 -2.30985776e-01 1.51839122e-...,[-4.22412418e-02 1.14388533e-01 3.38180526e-...
2497,James was at peace. His life with her had been...,1,Humano,2497,-12.964897,-2.644980,-2.242123,-2.618173,-0.134080,-0.046046,...,-0.046046,0.061846,-0.020167,0.000027,0.991123,[-0.20290406 -0.6610913 0.4014982 ... -1.05...,[ 0.32373595 -0.14913166 0.1407042 ... 0.60...,[ 1.5703125 -0.8515625 0.01147461 ... -0.17...,[-2.28550300e-01 -4.65069294e-01 -3.47584814e-...,[-8.68592933e-02 7.48332366e-02 2.91463477e-...
2498,Jensen loved fishing with his dad. Since he wa...,0,IA,2498,-14.614481,-2.644980,-2.242123,-2.618173,-0.097570,-0.001389,...,-0.001389,0.006324,-0.001389,0.000020,0.998866,[ 0.29198748 -1.024297 -0.83597463 ... -0.33...,[ 0.5804435 -0.08371511 -1.5411438 ... 0.95...,[ 2.0625 -1.609375 -1.578125 ... -1.82812

In [19]:
df_performance = pd.read_csv('df_performance_models_proachs.csv')
df_performance

,Modelo,Enfoque,Tiempo (s),VRAM Pico (GB),RAM usada (GB)
0,LLaDA,Sequence Score,398.081016,15.534767,0.058983
1,GPT,Sequence Score,100.754739,14.994935,0.010731
2,LLAMA,Sequence Score,359.512608,15.877870,0.003891
3,GPT3,Sequence Score,210.824698,16.800073,0.007000
4,BERT,Sequence Score,19.478747,13.941291,0.006760
5,RoBERTa,Sequence Score,20.221334,14.468471,0.002151
6,LLaDA,PAWN Extraction,5755.237497,18.191759,0.004391
7,GPT,PAWN Extraction,110.113407,15.299666,0.000549
8,LLaMA,PAWN Extraction,366.363019,15.723276,0.000214
9,GPT3,PAWN Extraction,220.163312,16.373202,0.000088
